In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from transformers import pipeline

In [2]:
df = pd.read_csv("../data/BTC-USD_news.csv")
df.head()

,Tytuł,Źródło,Data,Link
0,How to invest in cryptocurrency: A beginner's ...,finance.yahoo.com,2026-03-16 16:59,https://finance.yahoo.com/personal-finance/inv...
1,Analysts issue stark Bitcoin warning after lar...,thestreet.com,2026-03-27 21:19,https://www.thestreet.com/crypto/markets/analy...
2,Crypto Thefts Hit $2.7 Billion as Coinbase Cov...,finance.yahoo.com,2026-03-27 21:07,https://finance.yahoo.com/markets/crypto/artic...
3,Top Cryptocurrencies Fall; Bitcoin Falls Below...,finance.yahoo.com,2026-03-27 21:04,https://finance.yahoo.com/markets/crypto/artic...
4,Sector Update: Financial Stocks Decline Late A...,finance.yahoo.com,2026-03-27 21:01,https://finance.yahoo.com/markets/stocks/artic...


In [3]:
sentiment_model = pipeline("text-classification")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [4]:
titles = df["Tytuł"].tolist()
results = sentiment_model(titles)
df["sentiment_label"] = [result["label"] for result in results]
df["sentiment_score"] = [result["score"] for result in results]
df["sentiment_score"] = df["sentiment_score"].round(4)
df.head(10)

,Tytuł,Źródło,Data,Link,sentiment_label,sentiment_score
0,How to invest in cryptocurrency: A beginner's ...,finance.yahoo.com,2026-03-16 16:59,https://finance.yahoo.com/personal-finance/inv...,POSITIVE,0.9940
1,Analysts issue stark Bitcoin warning after lar...,thestreet.com,2026-03-27 21:19,https://www.thestreet.com/crypto/markets/analy...,NEGATIVE,0.9942
2,Crypto Thefts Hit $2.7 Billion as Coinbase Cov...,finance.yahoo.com,2026-03-27 21:07,https://finance.yahoo.com/markets/crypto/artic...,NEGATIVE,0.9975
3,Top Cryptocurrencies Fall; Bitcoin Falls Below...,finance.yahoo.com,2026-03-27 21:04,https://finance.yahoo.com/markets/crypto/artic...,NEGATIVE,0.9997
4,Sector Update: Financial Stocks Decline Late A...,finance.yahoo.com,2026-03-27 21:01,https://finance.yahoo.com/markets/stocks/artic...,NEGATIVE,0.9982
5,Bitcoin Drops 4.3% as $14 Billion Options Expi...,finance.yahoo.com,2026-03-27 21:00,https://finance.yahoo.com/markets/crypto/artic...,NEGATIVE,0.9994
6,Bitcoin Policy Institute Voices Strong Opposit...,bankless.com,2026-03-27 19:47,https://www.bankless.com/read/news/bitcoin-pol...,POSITIVE,0.7277
7,Analyst warns Bitcoin traders are buying ‘cras...,thestreet.com,2026-03-27 19:30,https://www.thestreet.com/crypto/markets/analy...,NEGATIVE,0.9901
8,Bitcoin extends slide as options point toward ...,finance.yahoo.com,2026-03-27 19:20,https://finance.yahoo.com/news/bitcoin-slumps-...,NEGATIVE,0.9913
9,Sector Update: Financial Stocks Decline Friday...,finance.yahoo.com,2026-03-27 19:04,https://finance.yahoo.com/markets/stocks/artic...,NEGATIVE,0.9987


In [ ]:
# Kierunek sentymentu + siła sentymentu
def callculate_sentiment_score(row):
    if row["sentiment_label"] == "POSITIVE":
        return row["sentiment_score"]
    elif row["sentiment_label"] == "NEGATIVE":
        return -row["sentiment_score"]
    else:
        return 0.0

# Warygodność żródła
def get_source_reliability(source):
    reliable_sources = ["finance.yahoo.com"]
    unreliable_sources = ["bankless.com"]
    if source in reliable_sources:
        return 1.0
    elif source in unreliable_sources:
        return 0.1
    else:
        return 0.5

# Aktualność informacji
def calculate_freshness_score(pub_date):
    current_time = pd.Timestamp.now()
    pub_time = pd.to_datetime(pub_date)
    time_diff = (current_time - pub_time).total_seconds() / 3600
    if time_diff < 1:
        return 1.0
    elif time_diff < 24:
        return 0.8
    elif time_diff < 72:
        return 0.5
    else:
        return 0.2


# Inportance score = 0.5 * sentiment_score + 0.3 * source_reliability + 0.2 * freshness_score
def calculate_importance_score(row):
    sentiment_score = callculate_sentiment_score(row)
    source_reliability = get_source_reliability(row["Źródło"])
    freshness_score = calculate_freshness_score(row["Data"])
    importance_score = (
        0.5 * sentiment_score + 0.3 * source_reliability + 0.2 * freshness_score
    )
    return importance_score

# если сентимент скор ниже нуля то весь импортанс скор должен быть ниже нуля

df["importance_score"] = df.apply(calculate_importance_score, axis=1)
df.head(10)

,Tytuł,Źródło,Data,Link,sentiment_label,sentiment_score,importance_score
0,How to invest in cryptocurrency: A beginner's ...,finance.yahoo.com,2026-03-16 16:59,https://finance.yahoo.com/personal-finance/inv...,POSITIVE,0.9940,0.83700
1,Analysts issue stark Bitcoin warning after lar...,thestreet.com,2026-03-27 21:19,https://www.thestreet.com/crypto/markets/analy...,NEGATIVE,0.9942,-0.18710
2,Crypto Thefts Hit $2.7 Billion as Coinbase Cov...,finance.yahoo.com,2026-03-27 21:07,https://finance.yahoo.com/markets/crypto/artic...,NEGATIVE,0.9975,-0.03875
3,Top Cryptocurrencies Fall; Bitcoin Falls Below...,finance.yahoo.com,2026-03-27 21:04,https://finance.yahoo.com/markets/crypto/artic...,NEGATIVE,0.9997,-0.03985
4,Sector Update: Financial Stocks Decline Late A...,finance.yahoo.com,2026-03-27 21:01,https://finance.yahoo.com/markets/stocks/artic...,NEGATIVE,0.9982,-0.03910
5,Bitcoin Drops 4.3% as $14 Billion Options Expi...,finance.yahoo.com,2026-03-27 21:00,https://finance.yahoo.com/markets/crypto/artic...,NEGATIVE,0.9994,-0.03970
6,Bitcoin Policy Institute Voices Strong Opposit...,bankless.com,2026-03-27 19:47,https://www.bankless.com/read/news/bitcoin-pol...,POSITIVE,0.7277,0.55385
7,Analyst warns Bitcoin traders are buying ‘cras...,thestreet.com,2026-03-27 19:30,https://www.thestreet.com/crypto/markets/analy...,NEGATIVE,0.9901,-0.18505
8,Bitcoin extends slide as options point toward ...,finance.yahoo.com,2026-03-27 19:20,https://finance.yahoo.com/news/bitcoin-slumps-...,NEGATIVE,0.9913,-0.03565
9,Sector Update: Financial Stocks Decline Friday...,finance.yahoo.com,2026-03-27 19:04,https://finance.yahoo.com/markets/stocks/artic...,NEGATIVE,0.9987,-0.03935
